# Convert ENVI Hyperspectral Data to Zarr

Due to large data block, we chunk into multiple regions across multiple wavelengths, in order to avoid loading entire image into memory. This notebook converts the SA1 ENVI hyperspectral mosaic (`.hdr` + `.dat`) into a chunked Zarr array.

In [10]:
import json

from pathlib import Path
import numpy as np
import zarr
import spectral.io.envi as envi
from tqdm import tqdm


data_dir = Path("../data/SA1_K4")
HDR_PATH = data_dir / "MOZ/SA1_K4_HS_MOZ.hdr"
DAT_PATH = data_dir / "MOZ/SA1_K4_HS_MOZ.dat"

ZARR_PATH = data_dir / "SA1_K4_HS_MOZ.zarr"
ZARR_META_PATH = data_dir/ "SA1_K4_HS_MOZ.zarr_metadata.json"

OVERWRITE_ZARR = False


### Convert to Zarr array


First, open a Zarr object, chunked to 512 x 512 x 16 blocks, which provides a practical balance between memory efficiency and read/write performance. Check first that file exists before overwriting. 


In [11]:
hs_img = envi.open(HDR_PATH, DAT_PATH)
rows, cols, bands = hs_img.shape

chunk_shape = (512, 512, 16)

zarr_already_exists = ZARR_PATH.exists()

if OVERWRITE_ZARR:
    print("Overwriting existing Zarr.")
    z = zarr.open(
        ZARR_PATH,
        mode="w",
        shape=(rows, cols, bands),
        chunks=chunk_shape,
        dtype="float32",
    )
    write_data = True

elif zarr_already_exists:
    print("Zarr already exists. Opening without rewriting.")
    z = zarr.open(ZARR_PATH, mode="r+")
    write_data = False

else:
    print("Creating new Zarr.")
    z = zarr.open(
        ZARR_PATH,
        mode="w",
        shape=(rows, cols, bands),
        chunks=chunk_shape,
        dtype="float32",
    )
    write_data = True

print("Zarr location:", ZARR_PATH)
print("Zarr shape:", z.shape)
print("Zarr chunks:", z.chunks)

Zarr already exists. Opening without rewriting.
Zarr location: ..\data\SA1_K4\SA1_K4_HS_MOZ.zarr
Zarr shape: (6896, 5496, 430)
Zarr chunks: (512, 512, 16)


Copy the ENVI header metadata into Zarr attributes and write a JSON metadata backup next to the Zarr store. Later notebooks use the saved wavelength metadata for spectral plots and band selection.


In [12]:
metadata = dict(hs_img.metadata)

z.attrs["envi_header"] = metadata
z.attrs["source_hdr"] = str(HDR_PATH)
z.attrs["source_dat"] = str(DAT_PATH)
z.attrs["shape"] = [int(rows), int(cols), int(bands)]
z.attrs["chunks"] = list(chunk_shape)
z.attrs["dtype"] = "float32"

if "wavelength" in metadata:
    z.attrs["wavelength"] = [float(w) for w in metadata["wavelength"]]

if "bands" in metadata:
    z.attrs["bands"] = int(metadata["bands"])

if "interleave" in metadata:
    z.attrs["interleave"] = str(metadata["interleave"])

if "data type" in metadata:
    z.attrs["envi_data_type"] = str(metadata["data type"])

with open(ZARR_META_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Saved metadata JSON:", ZARR_META_PATH)

Saved metadata JSON: ..\data\SA1_K4\SA1_K4_HS_MOZ.zarr_metadata.json


Copy the hyperspectral cube from ENVI into Zarr


In [13]:
if write_data:
    band_chunk = 16

    for b0 in tqdm(range(0, bands, band_chunk), desc="Writing Zarr chunks", unit="band-block"):
        b1 = min(b0 + band_chunk, bands)

        band_stack = np.empty((rows, cols, b1 - b0), dtype=np.float32)

        for i, b in enumerate(range(b0, b1)):
            band_stack[:, :, i] = hs_img.read_band(b).astype(np.float32, copy=False)

        z[:, :, b0:b1] = band_stack

        del band_stack

    z.attrs["data_written"] = True
    print("Finished writing data to Zarr.")

else:
    print("Skipping data copy because Zarr already exists.")

Skipping data copy because Zarr already exists.


Verify Output

In [14]:
print("Zarr conversion complete.")
print("Zarr shape:", z.shape)
print("Zarr dtype:", z.dtype)
print("Zarr chunks:", z.chunks)
print("Stored attributes:", sorted(z.attrs.keys()))
print("Sample value:", z[0, 0, 0])


Zarr conversion complete.
Zarr shape: (6896, 5496, 430)
Zarr dtype: float32
Zarr chunks: (512, 512, 16)
Stored attributes: ['bands', 'chunks', 'dtype', 'envi_data_type', 'envi_header', 'interleave', 'shape', 'source_dat', 'source_hdr', 'wavelength']
Sample value: 0.0144
